# Debug via UART en sistemas FreeRTOS — STM32 Nucleo F103RB

**Scope:** Guía práctica de debugging con UART para sistemas embebidos con FreeRTOS.  
**Hardware:** STM32 Nucleo F103RB (STM32F103RB, Cortex-M3)  
**Herramientas:** STM32CubeIDE, HAL, terminal serie (PuTTY / minicom / Termite)


## 1. Concepto de UART

**UART** (Universal Asynchronous Receiver/Transmitter) es el protocolo serie más simple y universal en sistemas embebidos. Transmite datos bit a bit sin señal de reloj compartida — ambos extremos se ponen de acuerdo en la velocidad de antemano.

### Trama de un byte

```
IDLE  START  D0  D1  D2  D3  D4  D5  D6  D7  STOP  IDLE
  1     0    ─── datos (LSB primero) ───────    1     1
```

- **Start bit:** siempre 0 — avisa al receptor que llega un byte.
- **Data bits:** 8 bits de datos (configuración más común).
- **Stop bit:** siempre 1 — permite que el receptor se resincronice.
- **Sin reloj:** el receptor muestrea los bits según el baud rate acordado.

### Parámetros de configuración

| Parámetro | Valor típico | Descripción |
|-----------|-------------|-------------|
| Baud rate | 115200 bps | Velocidad en bits por segundo |
| Data bits | 8 | Bits por trama |
| Paridad | None | Sin bit de paridad |
| Stop bits | 1 | Un stop bit |
| Flow control | None | Sin control de flujo hardware |

Esta configuración se abrevia como **115200 8N1** y es el estándar de facto para debug.

### ¿Por qué UART para debug?

- Solo necesita **2 pines** (TX y RX) o incluso 1 (solo TX para imprimir).
- El STM32 Nucleo F103RB incluye un **conversor USB-Serial integrado** en el ST-Link — no necesita hardware adicional.
- Es el mecanismo más bajo nivel disponible: funciona incluso si el sistema está colgado, no requiere sistema de archivos, ni display, ni red.
- Overhead mínimo en el sistema.


## 2. Conexión Física y Herramientas

### STM32 Nucleo F103RB — UART integrado

El Nucleo F103RB tiene un ST-Link/V2 integrado que incluye un **Virtual COM Port (VCP)**: un conversor USB-Serial que aparece como puerto COM en la PC al conectar el USB.

```
STM32F103RB          ST-Link (integrado)       PC
────────────         ──────────────────         ──
PA2 (USART2 TX) ──→  ST-Link RX        ──USB──→ COM port
PA3 (USART2 RX) ←──  ST-Link TX                 COM port
```

**En STM32CubeMX / CubeIDE:** habilitar `USART2` en modo Asynchronous. Los pines PA2/PA3 se configuran automáticamente.

### Configuración en CubeMX

```
Connectivity → USART2
  Mode: Asynchronous
  Baud Rate: 115200
  Word Length: 8 Bits
  Parity: None
  Stop Bits: 1
```

Esto genera el handle `huart2` en el código.

### Herramientas de terminal en la PC

| Herramienta | OS | Destacado |
|-------------|----|-----------|
| **PuTTY** | Windows | Simple, confiable, muy usado |
| **Termite** | Windows | Muestra caracteres no imprimibles, timestamps |
| **CubeIDE Serial Monitor** | Todos | Integrado en el IDE |
| **minicom** | Linux/Mac | Terminal de línea de comandos |
| **screen** | Linux/Mac | `screen /dev/ttyUSB0 115200` |

### Identificar el puerto

- **Windows:** Device Manager → Ports (COM & LPT) → STMicroelectronics Virtual COM Port → COMx
- **Linux:** `ls /dev/ttyACM*` o `ls /dev/ttyUSB*` → generalmente `/dev/ttyACM0`
- **Mac:** `ls /dev/cu.usbmodem*`

### Alternativa: Conversor USB-TTL externo

Si se usan UARTs distintos del USART2 (por ejemplo USART1 en PA9/PA10), se necesita un conversor externo CP2102, CH340 o FTDI:

```
STM32          Conversor USB-TTL
───────        ─────────────────
TX (PA9)  ──→  RX del conversor
RX (PA10) ←──  TX del conversor
GND       ───  GND
```

> **Importante:** nunca conectar el 3.3V/5V del conversor al VCC del Nucleo cuando el Nucleo ya está alimentado por USB.


## 3. Implementación Clásica de `printf()` via UART

Por defecto, `printf()` en un proyecto STM32 HAL no va a ningún lado. Para redirigirlo al UART hay que **retarget** la función de bajo nivel `_write()` (o `fputc()` según el compilador).

### Paso 1: Configurar el linker (syscalls)

En proyectos generados por CubeIDE, el archivo `syscalls.c` ya existe. Solo hay que modificar `_write()`:

```c
/* En syscalls.c — redirigir _write() al UART */
#include "usart.h"   /* para huart2 */

extern UART_HandleTypeDef huart2;

int __io_putchar(int ch) {
    HAL_UART_Transmit(&huart2, (uint8_t *)&ch, 1, HAL_MAX_DELAY);
    return ch;
}

int _write(int file, char *ptr, int len) {
    HAL_UART_Transmit(&huart2, (uint8_t *)ptr, len, HAL_MAX_DELAY);
    return len;
}
```

### Paso 2: Habilitar el buffer de printf

En `main.c`, antes de usar printf, deshabilitar el buffering para que cada `printf()` se envíe inmediatamente:

```c
int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_USART2_UART_Init();

    /* Deshabilitar buffering de stdout */
    setvbuf(stdout, NULL, _IONBF, 0);

    printf("Sistema iniciado\r\n");
    /* ... */
}
```

### Paso 3: Uso básico

```c
/* Imprimir texto */
printf("Temperatura: %d C\r\n", temp);

/* Imprimir con timestamp */
printf("[%lu ms] Sensor leido: %d\r\n", HAL_GetTick(), valor);

/* Debug de variables flotantes (requiere habilitar float en linker) */
printf("Voltaje: %.2f V\r\n", voltaje);
```

> **Para flotantes en STM32CubeIDE:** Project → Properties → C/C++ Build → Settings → MCU GCC Linker → Miscellaneous → agregar `-u _printf_float`

### Alternativa: función de debug mínima sin `printf()`

Si `printf()` consume demasiado flash/stack, una función mínima:

```c
void debug_print(const char *str) {
    HAL_UART_Transmit(&huart2, (uint8_t *)str, strlen(str), HAL_MAX_DELAY);
}

/* Uso: */
debug_print("[INFO] Loop iniciado\r\n");
```


## 4. Optimización de Memoria

`printf()` es cómodo pero costoso. El STM32F103RB tiene **128 KB de Flash y 20 KB de RAM** — hay que usarlos con cuidado.

### Costo de `printf()`

| Función | Flash adicional aprox. | RAM (stack) |
|---------|----------------------|-------------|
| `printf()` completo | ~5-8 KB | ~1-2 KB |
| `printf()` sin float | ~3-5 KB | ~512 B |
| `sprintf()` solo | ~4-6 KB | ~512 B |
| `HAL_UART_Transmit` + strings literales | ~0 extra | Mínimo |

### Estrategias de reducción

**1. Evitar `printf()` en producción — usar macros condicionales:**

```c
/* En un header: debug.h */
#ifdef DEBUG_ENABLED
    #define DBG(fmt, ...) printf("[DBG] " fmt "\r\n", ##__VA_ARGS__)
#else
    #define DBG(fmt, ...)  /* nada — se elimina en compilación */
#endif

/* Uso: */
DBG("Tarea iniciada, prio=%d", uxTaskPriorityGet(NULL));
```

En release: definir `DEBUG_ENABLED` como 0 o no definirlo → todo el código de debug desaparece sin cambiar el source.

**2. Usar `snprintf()` con buffer estático:**

```c
/* Evitar stack dinámico — buffer estático compartido (proteger con mutex) */
static char debug_buf[64];

void debug_printf(const char *fmt, ...) {
    va_list args;
    va_start(args, fmt);
    vsnprintf(debug_buf, sizeof(debug_buf), fmt, args);
    va_end(args);
    HAL_UART_Transmit(&huart2, (uint8_t *)debug_buf, strlen(debug_buf), 100);
}
```

**3. Niveles de log:**

```c
#define LOG_LEVEL_ERROR  0
#define LOG_LEVEL_WARN   1
#define LOG_LEVEL_INFO   2
#define LOG_LEVEL_DEBUG  3

#define CURRENT_LOG_LEVEL LOG_LEVEL_INFO

#define LOG_ERROR(fmt, ...) if (CURRENT_LOG_LEVEL >= LOG_LEVEL_ERROR) printf("[ERR] " fmt "\r\n", ##__VA_ARGS__)
#define LOG_INFO(fmt, ...)  if (CURRENT_LOG_LEVEL >= LOG_LEVEL_INFO)  printf("[INF] " fmt "\r\n", ##__VA_ARGS__)
#define LOG_DEBUG(fmt, ...) if (CURRENT_LOG_LEVEL >= LOG_LEVEL_DEBUG) printf("[DBG] " fmt "\r\n", ##__VA_ARGS__)
```

**4. Imprimir solo datos relevantes:**

```c
/* Malo: imprime en cada ciclo (miles de líneas por segundo) */
for (;;) {
    printf("ADC = %d\r\n", HAL_ADC_GetValue(&hadc1));
}

/* Mejor: imprimir cada N ms */
static uint32_t last_print = 0;
if (HAL_GetTick() - last_print > 500) {
    printf("ADC = %d\r\n", HAL_ADC_GetValue(&hadc1));
    last_print = HAL_GetTick();
}
```


## 5. `printf()` en un Entorno Multitarea (FreeRTOS)

En un sistema con múltiples tareas, varias pueden llamar a `printf()` simultáneamente. Esto causa **corrupción de mensajes** porque `printf()` no es thread-safe.

### El problema: salida entrelazada

```
Tarea A: printf("Sensor: %d\r\n", val);
Tarea B: printf("Motor: ON\r\n");

Salida real en UART:
SeMotnsor:or: 42ON
              ↑ texto corrupto
```

### Solución 1: Mutex sobre printf (simple)

```c
#include "FreeRTOS.h"
#include "semphr.h"

static SemaphoreHandle_t xPrintMutex = NULL;

void uart_printf(const char *fmt, ...) {
    char buf[128];
    va_list args;
    va_start(args, fmt);
    vsnprintf(buf, sizeof(buf), fmt, args);
    va_end(args);

    if (xPrintMutex != NULL) {
        xSemaphoreTake(xPrintMutex, portMAX_DELAY);
        HAL_UART_Transmit(&huart2, (uint8_t *)buf, strlen(buf), 100);
        xSemaphoreGive(xPrintMutex);
    }
}

/* Inicializar en main() antes de vTaskStartScheduler() */
xPrintMutex = xSemaphoreCreateMutex();

/* Uso desde cualquier tarea: */
uart_printf("[Task A] Temperatura: %d C\r\n", temp);
```

### Solución 2: Cola de mensajes (no bloquea la tarea que imprime)

Una tarea dedicada a UART recibe mensajes de una cola y los imprime. Las tareas de aplicación nunca bloquean esperando al UART.

```c
#define LOG_QUEUE_LENGTH  10
#define LOG_MSG_SIZE      80

typedef char LogMessage_t[LOG_MSG_SIZE];
static QueueHandle_t xLogQueue;

/* Tarea impresora — prioridad baja */
static void vUartPrintTask(void *pvParams) {
    LogMessage_t msg;
    for (;;) {
        xQueueReceive(xLogQueue, &msg, portMAX_DELAY);
        HAL_UART_Transmit(&huart2, (uint8_t *)msg, strlen(msg), 200);
    }
}

/* Función de log que usan las tareas — no bloquea (xTicksToWait=0) */
void log_send(const char *fmt, ...) {
    LogMessage_t buf;
    va_list args;
    va_start(args, fmt);
    vsnprintf(buf, LOG_MSG_SIZE, fmt, args);
    va_end(args);
    xQueueSend(xLogQueue, &buf, 0);  /* no bloquear si la cola está llena */
}

/* En main(): */
xLogQueue = xQueueCreate(LOG_QUEUE_LENGTH, sizeof(LogMessage_t));
xTaskCreate(vUartPrintTask, "UartLog", 256, NULL, 1, NULL);
```

**Ventaja:** las tareas de aplicación llaman a `log_send()` y continúan sin esperar al UART. Si la cola se llena, el mensaje se descarta (controlable).

### Agregar timestamp automático

```c
void log_send(const char *fmt, ...) {
    LogMessage_t buf;
    int offset = snprintf(buf, LOG_MSG_SIZE, "[%6lu] ", xTaskGetTickCount());
    va_list args;
    va_start(args, fmt);
    vsnprintf(buf + offset, LOG_MSG_SIZE - offset, fmt, args);
    va_end(args);
    xQueueSend(xLogQueue, &buf, 0);
}

/* Salida: */
/* [  1234] [SensorTask] Temperatura: 25 C */
/* [  1250] [MotorTask] Motor: ON          */
```


## 6. Recepción de Comandos por UART (UART RX)

Además de imprimir, el UART permite **enviar comandos desde la PC al STM32** — muy útil para controlar el sistema durante el desarrollo sin necesidad de recompilar.

Hay tres modos de recepción, cada uno con sus ventajas:

| Modo | CPU usage | Latencia | Complejidad | Recomendado para |
|------|-----------|----------|-------------|------------------|
| Polling | Alto (bloquea) | Alta | Baja | Pruebas rápidas sin RTOS |
| Interrupción | Bajo | Baja | Media | Mayoría de aplicaciones RTOS |
| DMA | Mínimo | Mínima | Alta | Alto baudrate, mucho volumen |

---

### 6.1 Modo Polling

La tarea llama a `HAL_UART_Receive()` que **bloquea** hasta recibir los bytes solicitados o hasta que expire el timeout.

```c
static void vCommandTask(void *pvParams) {
    uint8_t rx_byte;
    char    cmd_buf[32];
    int     idx = 0;

    for (;;) {
        /* Espera 1 byte, timeout 100ms */
        if (HAL_UART_Receive(&huart2, &rx_byte, 1, 100) == HAL_OK) {
            if (rx_byte == '\n' || rx_byte == '\r') {
                cmd_buf[idx] = '\0';
                vProcessCommand(cmd_buf);
                idx = 0;
            } else if (idx < sizeof(cmd_buf) - 1) {
                cmd_buf[idx++] = rx_byte;
            }
        }
        /* Si no llega nada en 100ms, el bloqueo termina y
           otras tareas pueden ejecutar */
    }
}
```

**Limitaciones del polling:**
- La tarea queda bloqueada durante el timeout (100 ms en el ejemplo). Con FreeRTOS esto no consume CPU — la tarea está en Blocked — pero si el timeout es 0, es un busy-wait.
- No apto para altas velocidades o flujos continuos de datos.

---

### 6.2 Modo Interrupción

Cada byte recibido genera una interrupción. La ISR notifica a una tarea FreeRTOS mediante un semáforo o cola. La tarea procesa el byte cuando es su turno.

```c
static QueueHandle_t xUartRxQueue;
static uint8_t       rx_byte_isr;  /* buffer para la ISR */

/* Inicialización — activar recepción por interrupción */
void vUartRxInit(void) {
    xUartRxQueue = xQueueCreate(64, sizeof(uint8_t));
    /* Activar recepción del primer byte por IRQ */
    HAL_UART_Receive_IT(&huart2, &rx_byte_isr, 1);
}

/* Callback de HAL — ejecuta en contexto de ISR */
void HAL_UART_RxCpltCallback(UART_HandleTypeDef *huart) {
    if (huart->Instance == USART2) {
        BaseType_t xHigherPriorityTaskWoken = pdFALSE;

        /* Enviar el byte recibido a la cola */
        xQueueSendFromISR(xUartRxQueue, &rx_byte_isr, &xHigherPriorityTaskWoken);

        /* Reactivar recepción del siguiente byte */
        HAL_UART_Receive_IT(&huart2, &rx_byte_isr, 1);

        portYIELD_FROM_ISR(xHigherPriorityTaskWoken);
    }
}

/* Tarea que procesa los bytes recibidos */
static void vUartRxTask(void *pvParams) {
    uint8_t byte;
    char    cmd_buf[32];
    int     idx = 0;

    for (;;) {
        /* Esperar hasta recibir un byte de la cola */
        xQueueReceive(xUartRxQueue, &byte, portMAX_DELAY);

        if (byte == '\r' || byte == '\n') {
            cmd_buf[idx] = '\0';
            if (idx > 0) vProcessCommand(cmd_buf);
            idx = 0;
        } else if (idx < sizeof(cmd_buf) - 1) {
            cmd_buf[idx++] = byte;
        }
    }
}
```

**Ventajas:** CPU libre entre caracteres, latencia baja, funciona bien con FreeRTOS.

---

### 6.3 Modo DMA

El DMA transfiere datos del periférico UART directamente a RAM sin intervención de la CPU. Solo se genera una interrupción al finalizar el bloque.

```c
#define RX_BUF_SIZE 64
static uint8_t rx_dma_buf[RX_BUF_SIZE];
static SemaphoreHandle_t xDmaRxSemaphore;

void vUartDmaInit(void) {
    xDmaRxSemaphore = xSemaphoreCreateBinary();
    /* Recibir hasta RX_BUF_SIZE bytes por DMA */
    HAL_UART_Receive_DMA(&huart2, rx_dma_buf, RX_BUF_SIZE);
}

/* Callback de HAL — ejecuta en ISR cuando DMA completa */
void HAL_UART_RxCpltCallback(UART_HandleTypeDef *huart) {
    if (huart->Instance == USART2) {
        BaseType_t xHigherPriorityTaskWoken = pdFALSE;
        xSemaphoreGiveFromISR(xDmaRxSemaphore, &xHigherPriorityTaskWoken);
        portYIELD_FROM_ISR(xHigherPriorityTaskWoken);
    }
}

static void vDmaRxTask(void *pvParams) {
    for (;;) {
        /* Esperar hasta que DMA llene el buffer */
        xSemaphoreTake(xDmaRxSemaphore, portMAX_DELAY);
        /* Procesar rx_dma_buf[0..RX_BUF_SIZE-1] */
        vProcessBuffer(rx_dma_buf, RX_BUF_SIZE);
        /* Reactivar DMA */
        HAL_UART_Receive_DMA(&huart2, rx_dma_buf, RX_BUF_SIZE);
    }
}
```

**Nota DMA en STM32F103:** el DMA1 es compartido por varios periféricos. USART2 RX usa el **canal 6 del DMA1**. Hay que habilitarlo en CubeMX: USART2 → DMA Settings → Add → USART2_RX.

**Variante útil: UART idle line detection + DMA**  
En lugar de esperar N bytes fijos, se usa la interrupción de idle line (silencio en el bus) para detectar el fin de un mensaje de longitud variable:

```c
/* Activar IDLE line interrupt + DMA */
HAL_UARTEx_ReceiveToIdle_DMA(&huart2, rx_dma_buf, RX_BUF_SIZE);

/* Callback cuando llegan bytes y hay silencio en el bus */
void HAL_UARTEx_RxEventCallback(UART_HandleTypeDef *huart, uint16_t Size) {
    /* Size = número de bytes efectivamente recibidos */
    vProcessBuffer(rx_dma_buf, Size);
    HAL_UARTEx_ReceiveToIdle_DMA(&huart2, rx_dma_buf, RX_BUF_SIZE);
}
```


## 7. Conflictos Comunes entre HAL y FreeRTOS (Troubleshooting)

La combinación HAL + FreeRTOS tiene varias trampas conocidas. Esta sección describe las más frecuentes y cómo resolverlas.

---

### 7.1 `HAL_Delay()` bloquea el scheduler

**Problema:** `HAL_Delay()` usa el SysTick en un busy-wait. Cuando se llama desde una tarea FreeRTOS, **bloquea la CPU** completa — ninguna otra tarea puede ejecutar durante ese tiempo.

```c
/* MAL — congela todo el sistema durante 1 segundo */
HAL_Delay(1000);

/* BIEN — libera la CPU durante 1 segundo */
vTaskDelay(pdMS_TO_TICKS(1000));
```

**Regla:** nunca usar `HAL_Delay()` desde una tarea FreeRTOS. Siempre `vTaskDelay()` o `vTaskDelayUntil()`.

---

### 7.2 SysTick compartido — conflicto de prioridades

**Problema:** tanto HAL como FreeRTOS usan el SysTick. FreeRTOS toma control del SysTick para el tick del scheduler, dejando a HAL sin su fuente de tiempo si no se configura correctamente.

**Solución en CubeMX:**  
En proyectos con FreeRTOS: `System Core → SYS → Timebase Source` → cambiar de **SysTick** a **TIM1** (o cualquier timer disponible). CubeIDE muestra un warning que guía esto.

```c
/* Después del cambio, HAL usa TIM1 internamente.    */
/* FreeRTOS sigue usando SysTick para su tick.       */
/* HAL_GetTick() y vTaskDelay() coexisten sin problemas. */
```

---

### 7.3 `HAL_UART_Transmit()` en modo blocking desde múltiples tareas

**Problema:** si dos tareas llaman a `HAL_UART_Transmit()` simultáneamente, los bytes se entrelazan y ambas operaciones pueden corromperse.

**Solución:** proteger con un mutex (ver Sección 5) o usar la tarea impresora con cola.

---

### 7.4 `HAL_UART_Receive_IT()` necesita reactivarse en cada callback

**Problema:** `HAL_UART_Receive_IT()` configura una recepción de N bytes y al completarla se **desactiva automáticamente**. Si no se reactiva en el callback, el UART queda sordo.

```c
void HAL_UART_RxCpltCallback(UART_HandleTypeDef *huart) {
    /* Procesar byte... */
    /* SIEMPRE reactivar: */
    HAL_UART_Receive_IT(&huart2, &rx_byte_isr, 1);
}
```

---

### 7.5 Stack overflow de tareas

**Problema:** `printf()` y `vsnprintf()` consumen stack significativo. Si el stack de la tarea es muy pequeño, FreeRTOS puede detectar un stack overflow (o no, causando comportamiento impredecible).

**Solución:**

```c
/* Habilitar detección de stack overflow en FreeRTOSConfig.h */
#define configCHECK_FOR_STACK_OVERFLOW  2

/* Implementar el hook de stack overflow */
void vApplicationStackOverflowHook(TaskHandle_t xTask, char *pcTaskName) {
    /* Aquí el sistema detectó overflow — loguear y detener */
    HAL_UART_Transmit(&huart2, (uint8_t *)"STACK OVERFLOW!\r\n", 17, 100);
    for (;;);  /* detener el sistema */
}
```

Para tareas que usan `printf()`, asignar al menos **512 bytes** de stack.

---

### 7.6 Interrupciones con prioridad mayor que `configMAX_SYSCALL_INTERRUPT_PRIORITY`

**Problema:** si una ISR (como la del UART) tiene prioridad numérica **menor** que `configMAX_SYSCALL_INTERRUPT_PRIORITY` (en Cortex-M, menor número = mayor prioridad), llamar a APIs de FreeRTOS desde esa ISR causa un Hard Fault.

```c
/* En FreeRTOSConfig.h: */
#define configMAX_SYSCALL_INTERRUPT_PRIORITY  5  /* NVIC priority 5 */

/* Las ISRs que llaman a FreeRTOS API deben tener prioridad >= 5 (numéricamente) */
/* Es decir, prioridad NVIC 5, 6, 7, ... (MENOR urgencia) */
/* NUNCA asignar prioridad 0, 1, 2, 3, 4 a ISRs que usen FreeRTOS */
```

En CubeMX, verificar en `NVIC` que la prioridad de `USART2_IRQn` sea >= `configMAX_SYSCALL_INTERRUPT_PRIORITY`.


## 8. Visualización del Comportamiento (Tracing)

Cuando el sistema es complejo, los mensajes de texto no son suficientes. Hay herramientas de tracing que permiten ver la ejecución de tareas en el tiempo.

---

### 8.1 Tracing Manual con UART (mínimo esfuerzo)

La técnica más simple: imprimir un carácter o símbolo al entrar/salir de secciones clave. Útil para verificar la secuencia de ejecución.

```c
static void vSensorTask(void *pvParams) {
    for (;;) {
        printf(">S");      /* Tarea sensor: inicio ciclo */
        vLeerSensor();
        printf("<S\r\n"); /* Tarea sensor: fin ciclo */
        vTaskDelay(pdMS_TO_TICKS(100));
    }
}

static void vMotorTask(void *pvParams) {
    for (;;) {
        printf(">M");
        vControlarMotor();
        printf("<M\r\n");
        vTaskDelay(pdMS_TO_TICKS(50));
    }
}

/* Salida: >S<S >M<M >M<M >S<S ...   */
/* Permite ver qué tarea ejecuta cuándo */
```

---

### 8.2 FreeRTOS Runtime Stats

FreeRTOS puede registrar el **tiempo de CPU** que consume cada tarea. Se imprime con `vTaskGetRunTimeStats()`.

```c
/* En FreeRTOSConfig.h: */
#define configGENERATE_RUN_TIME_STATS          1
#define configUSE_STATS_FORMATTING_FUNCTIONS   1
#define portCONFIGURE_TIMER_FOR_RUN_TIME_STATS()  vConfigureTimerForRunTimeStats()
#define portGET_RUN_TIME_COUNTER_VALUE()          ulGetRunTimeCounterValue()

/* Función que imprime las estadísticas: */
void vPrintTaskStats(void) {
    static char buf[512];
    vTaskGetRunTimeStats(buf);
    printf("\r\nTask Name       Abs Time   %%Time\r\n");
    printf("-------------------------------\r\n");
    printf("%s", buf);
}
```

**Salida de ejemplo:**
```
Task Name       Abs Time   %Time
-------------------------------
SensorTask      12450      62%%
MotorTask        4820      24%%
UartLog          1230       6%%
IDLE             1500       8%%
```

Esto permite detectar tareas que consumen demasiada CPU.

---

### 8.3 FreeRTOS Task List

Ver el estado actual de todas las tareas:

```c
/* En FreeRTOSConfig.h: */
#define configUSE_TRACE_FACILITY               1
#define configUSE_STATS_FORMATTING_FUNCTIONS   1

void vPrintTaskList(void) {
    static char buf[512];
    vTaskList(buf);
    printf("\r\nNombre          Estado  Prio  Stack  Num\r\n");
    printf("------------------------------------------\r\n");
    printf("%s", buf);
}
```

**Salida de ejemplo:**
```
Nombre          Estado  Prio  Stack  Num
------------------------------------------
SensorTask      R       2     312    2
MotorTask       B       3     298    3
UartLog         B       1     401    4
IDLE            R       0     108    1
```

**Estados:** R=Running/Ready, B=Blocked, S=Suspended, D=Deleted  
**Stack:** mínimo de bytes libres restantes en el stack (menor = más riesgo de overflow)

---

### 8.4 Tracealyzer (herramienta visual, avanzada)

**Percepio Tracealyzer** es una herramienta de tracing visual para FreeRTOS que muestra un diagrama de Gantt de la ejecución de las tareas en tiempo real.

```
[PC] Tracealyzer ←── J-Link/ST-Link ←── STM32 (con librería TraceRecorder)
```

- Requiere agregar la librería `TraceRecorder` al proyecto.
- Versión gratuita disponible para proyectos académicos.
- Muestra context switches, semáforos, colas, y tiempo en cada estado.
- Muy útil para diagnosticar problemas de timing que son imposibles de ver con printf.

---

### 8.5 GPIO Toggle como señal de osciloscopio

Cuando el timing importa a nivel de microsegundos, el UART es demasiado lento. Togglear un GPIO y medirlo con osciloscopio o analizador lógico es la técnica más precisa:

```c
/* Al entrar a la tarea de mayor prioridad */
HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET);   /* LED ON */
vTareaTimeCritical();
HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_RESET); /* LED OFF */

/* En el osciloscopio: el pulso HIGH muestra exactamente cuánto dura la tarea */
```

Con un analizador lógico de 8 canales (como el Logic8 de Saleae o un clone barato) se pueden ver 8 señales simultáneas — una por tarea — y observar visualmente el scheduling.


## 9. Guía Rápida — Checklist de Debug

### Al comenzar un proyecto

- [ ] Verificar que el ST-Link VCP aparece como puerto COM en la PC.
- [ ] Configurar USART2 en CubeMX: 115200, 8N1, sin flow control.
- [ ] Cambiar el Timebase de HAL de SysTick a TIM1 (necesario con FreeRTOS).
- [ ] Agregar el retarget de `printf()` en `syscalls.c`.
- [ ] Agregar `setvbuf(stdout, NULL, _IONBF, 0)` en `main()` antes del scheduler.
- [ ] Habilitar `configCHECK_FOR_STACK_OVERFLOW 2` y el hook correspondiente.
- [ ] Crear un mutex o cola para proteger el acceso al UART desde múltiples tareas.

### Cuando algo falla

| Síntoma | Causa probable | Solución |
|---------|---------------|----------|
| No llega nada al terminal | COM incorrecto o UART no configurado | Verificar puerto COM, revisar MX_USART2_Init |
| Texto corrupto / mezclado | Múltiples tareas usan printf sin protección | Agregar mutex o cola de log |
| El sistema se cuelga al usar printf | Stack overflow | Aumentar stack de la tarea, habilitar stack overflow hook |
| Hard Fault en la ISR de UART | Prioridad de IRQ muy alta para FreeRTOS | Subir el número de prioridad NVIC de USART2_IRQ |
| UART RX deja de recibir | No se reactivó `Receive_IT` en el callback | Agregar `HAL_UART_Receive_IT()` al final del callback |
| `HAL_GetTick()` se congela | SysTick tomado por FreeRTOS, HAL sin timebase | Cambiar timebase de HAL a TIM1 en CubeMX |
| Otras tareas no corren durante un delay | Se usó `HAL_Delay()` en lugar de `vTaskDelay()` | Reemplazar por `vTaskDelay(pdMS_TO_TICKS(ms))` |

### Comandos UART útiles para debug remoto

```c
/* Ejemplo de mini-consola de comandos via UART */
void vProcessCommand(const char *cmd) {
    if (strcmp(cmd, "stats") == 0) {
        vPrintTaskStats();            /* ver uso de CPU por tarea */
    } else if (strcmp(cmd, "list") == 0) {
        vPrintTaskList();             /* ver estado de todas las tareas */
    } else if (strcmp(cmd, "reset") == 0) {
        NVIC_SystemReset();           /* reset por software */
    } else if (strcmp(cmd, "heap") == 0) {
        printf("Free heap: %d bytes\r\n", xPortGetFreeHeapSize());
    } else {
        printf("Comandos: stats | list | reset | heap\r\n");
    }
}
```

Con este patrón se puede interrogar el sistema en tiempo de ejecución sin detener el programa ni conectar un debugger.
